# MIstral 7b — sft pathology extraction

Runs **base** `mistralai/Mixtral-8x7B-Instruct-v0.1` for pathology-report JSON extraction.
With adapter sft


## 1. Imports and paths

In [1]:
import os
import json
import math
import random
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

try:
    from peft import PeftModel
except ImportError:
    PeftModel = None



In [2]:


MODEL_ID_OR_PATH = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_DIR = Path("models/mistral-7b-instruct-v0.3-pathology-lora-rk2_mgrf/")
RUN_MODE = "lora_rk2_mgrf"



USE_LORA_ADAPTER = True


DEVICE_MAP = "auto"
MAX_NEW_TOKENS = 256
DTYPE = "auto"
BATCH_SIZE = 10

OUTPUT_DIR = Path(f"outputInf_mistral7b_{RUN_MODE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_MODE:", RUN_MODE)
print("USE_LORA_ADAPTER:", USE_LORA_ADAPTER)
print("OUTPUT_DIR:", OUTPUT_DIR)


RUN_MODE: lora_rk2_mgrf
USE_LORA_ADAPTER: True
OUTPUT_DIR: outputInf_mistral7b_lora_rk2_mgrf


## 2. Load the evaluation dataframe

This matches your MedGemma notebook's data-loading pattern. Adjust paths if needed.

In [1]:
df = pd.read_json('./data/gold_data.json')


In [4]:
gold_cols = [c for c in df.columns if c != "content"]
gold_cols

['echantillon',
 'numero_gang_prel',
 'taille',
 'diagnostic',
 'grade',
 'emboles',
 'numero_gang_att',
 'ki67',
 're',
 'rp',
 'her2',
 'marges_saines',
 'rupture_capsulaire']

## 3. Field schema and task instruction

In [5]:
FIELD_ORDER = [
    "echantillon",
    "numero_gang_prel",
    "numero_gang_att",
    "taille",
    "diagnostic",
    "grade",
    "emboles",
    "ki67",
    "re",
    "rp",
    "her2",
    "marges_saines",
    "rupture_capsulaire",
]

NUMERIC_FIELDS = {"numero_gang_prel", "numero_gang_att", "taille", "ki67"}

SYSTEM_PROMPT = """
Tu es un système d'extraction d'information à partir de comptes rendus anatomopathologiques de cancer du sein.

Retourne uniquement un objet JSON valide.
N'invente aucune information.

Si une information est absente, ambiguë ou non déductible:
- utilise "unknown" pour les variables catégorielles
- utilise null pour les variables numériques.

Variables à extraire:
- echantillon: mammectomie, tumorectomie, curage ganglionnaire, biopsie, exérèse chirurgicale, unknown
- numero_gang_prel: nombre de ganglions prélevés/examinés, integer ou null
- numero_gang_att: nombre de ganglions atteints/envahis, integer ou null
- taille: taille tumorale maximale en mm, integer ou null
- diagnostic: histologie principale, string
- grade: SBR1, SBR2, SBR3, unknown
- emboles: oui, non, unknown
- ki67: pourcentage Ki67, integer ou null
- re: positifs, négatifs, unknown
- rp: positifs, négatifs, unknown
- her2: négatif, douteux, positif, unknown
- marges_saines: oui, non, unknown
- rupture_capsulaire: oui, non, unknown

Règles:
- Convertir les cm en mm pour taille.
- HER2 0 ou 1+ = négatif.
- HER2 2+ = douteux.
- HER2 3+ ou amplification FISH/CISH/SISH = positif.
- RE/RP positifs si expression positive, même faible.
- Marges R0/non envahies/saines = oui.
- Marges atteintes/R1/envahies = non.
""".strip()

In [6]:
shot_indices=[]
FEW_SHOTS=[]

## 5. JSON parser

In [7]:
def extract_first_json_object(text: str) -> Optional[Dict[str, Any]]:
    """Best-effort extraction of the first JSON object from model output."""
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    in_string = False
    escape = False

    for idx in range(start, len(text)):
        char = text[idx]

        if escape:
            escape = False
            continue
        if char == "\\":
            escape = True
            continue
        if char == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:idx + 1]
                try:
                    parsed = json.loads(candidate)
                    return parsed if isinstance(parsed, dict) else None
                except json.JSONDecodeError:
                    return None

    return None

## 6. Prompt builder 

In [8]:
# Retourne uniquement le JSON."""
def build_report_user_message(report: str) -> str:
    return f"""{SYSTEM_PROMPT}

Compte rendu:
\"\"\"
{report}
\"\"\"

Retourne uniquement le JSON."""


def build_mixtral_fewshot_prompt(
    report: str,
    tokenizer: Any,
    few_shots: List[Dict[str, Any]] = FEW_SHOTS,
) -> str:
    messages = []

    for ex in few_shots:
        messages.append({
            "role": "user",
            "content": build_report_user_message(ex["report"]),
        })
        messages.append({
            "role": "assistant",
            "content": json.dumps(
                ordered_label(ex["label"]),
                ensure_ascii=False,
                separators=(",", ":"),
            ),
        })

    messages.append({
        "role": "user",
        "content": build_report_user_message(report),
    })

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

## 7. Load tokenizer and model

In [1]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_OR_PATH, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_OR_PATH,
    torch_dtype=DTYPE,
    device_map=DEVICE_MAP,
)

if USE_LORA_ADAPTER:
    if PeftModel is None:
        raise ImportError("peft is required for RUN_MODE using LoRA. Install with: pip install peft")
    if not ADAPTER_DIR.exists():
        raise FileNotFoundError(f"LoRA adapter not found: {ADAPTER_DIR}")
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
    print(f"Loaded LoRA adapter from: {ADAPTER_DIR}")

model.eval()



In [10]:
def get_input_device(model: Any) -> torch.device:
    """Choose a safe device for input tensors when using device_map='auto'."""
    try:
        return model.device
    except Exception:
        return next(model.parameters()).device


def count_prompt_tokens(report: str, tokenizer: Any, few_shots: List[Dict[str, Any]] = FEW_SHOTS) -> int:
    prompt = build_mixtral_fewshot_prompt(report, tokenizer, few_shots=few_shots)
    return len(tokenizer(prompt, add_special_tokens=False)["input_ids"])

# Quick check on the first report.
count_prompt_tokens(df["content"].iloc[0], tokenizer)

1254

## 8. Single-report inference

In [11]:
@torch.inference_mode()
def generate_mixtral_extraction(
    report: str,
    model: Any,
    tokenizer: Any,
    few_shots: List[Dict[str, Any]] = FEW_SHOTS,
    max_new_tokens: int = MAX_NEW_TOKENS,
    return_raw: bool = True,
    return_prompt: bool = False,
) -> Dict[str, Any]:
    prompt = build_mixtral_fewshot_prompt(report, tokenizer, few_shots=few_shots)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=False,
    )
    inputs = {k: v.to(get_input_device(model)) for k, v in inputs.items()}

    generation_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    output_ids = model.generate(
        **inputs,
        generation_config=generation_config,
    )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    prediction = extract_first_json_object(raw_text)

    result = {"prediction": prediction}
    if return_raw:
        result["raw_text"] = raw_text
    if return_prompt:
        result["prompt"] = prompt
    return result

In [2]:
result_one = generate_mixtral_extraction(
    report=df["content"].iloc[0],
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    return_raw=True,
    return_prompt=True,
)

result_one["prediction"]

## 9. Batch inference

In [13]:
@torch.inference_mode()
def generate_mixtral_extraction_batch(
    reports: List[str],
    model: Any,
    tokenizer: Any,
    few_shots: List[Dict[str, Any]] = FEW_SHOTS,
    batch_size: int = BATCH_SIZE,
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> List[Dict[str, Any]]:
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    results: List[Dict[str, Any]] = []
    input_device = get_input_device(model)

    try:
        for start in tqdm(range(0, len(reports), batch_size), desc="Running Mixtral batch inference"):
            batch_reports = reports[start:start + batch_size]
            prompts = [
                build_mixtral_fewshot_prompt(report, tokenizer, few_shots=few_shots)
                for report in batch_reports
            ]

            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=False,
            )
            inputs = {k: v.to(input_device) for k, v in inputs.items()}

            generation_config = GenerationConfig(
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            output_ids = model.generate(
                **inputs,
                generation_config=generation_config,
            )

            prompt_width = inputs["input_ids"].shape[-1]
            for i, report in enumerate(batch_reports):
                generated_ids = output_ids[i][prompt_width:]
                raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
                pred = extract_first_json_object(raw_text)
                results.append({
                    "raw_text": raw_text,
                    "prediction": pred,
                })

    finally:
        tokenizer.padding_side = old_padding_side

    return results

## 10. Run Mistral

In [2]:
reports = df["content"].astype(str).tolist()
batch_outputs = generate_mixtral_extraction_batch(
    reports=reports,
    model=model,
    tokenizer=tokenizer,
    few_shots=FEW_SHOTS,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
)

results_mixtral = []
for i, (report, out) in enumerate(zip(reports, batch_outputs)):
    gold_variables = df.iloc[i][gold_cols].to_dict()
    results_mixtral.append({
        "row_id": i,
        "report": report,
        "gold_variables": gold_variables,
        "prediction": out["prediction"],
        "raw_text": out["raw_text"],
    })

results_mixtral_df = pd.DataFrame(results_mixtral)
results_mixtral_df.head()

In [16]:
results_mixtral_df.to_csv(OUTPUT_DIR / "results_sft.csv", index=False)
results_mixtral_df.to_json(
    OUTPUT_DIR / "results_sft.json",
    orient="records",
    force_ascii=False,
    indent=2,
)

OUTPUT_DIR

PosixPath('outputInf_mistral7b_lora_rk2_mgrf')